# Data Explorer: General Tabular Data QA

This notebook demonstrates the **general-purpose QA agent** -- it works with **any dataset** and **any question**, with no prior knowledge or domain-specific helpers.

Unlike the DABStep demo (which uses a distilled `helper.py` tuned for financial payments), this agent:
- Scans your data directory to discover files
- Explores the schema autonomously
- Writes Python code from scratch to answer your question

This is the same architecture that would be used when applying Data Explorer to a **new, unseen dataset**.

## Setup

In [ ]:
import os
import sys
import shutil
import subprocess
import pathlib
import asyncio
import time
from dotenv import load_dotenv

# Find repo root (works on both Brev and Docker)
repo_root = None
_result = subprocess.run(['git', 'rev-parse', '--show-toplevel'], capture_output=True, text=True)
if _result.returncode == 0:
    repo_root = _result.stdout.strip()
if not repo_root:
    _p = pathlib.Path.cwd()
    for _dir in [_p] + list(_p.parents):
        if (_dir / 'pyproject.toml').exists():
            repo_root = str(_dir)
            break
if not repo_root:
    for _c in [pathlib.Path.home() / 'data-explorer-launchable', pathlib.Path('/app')]:
        if (_c / 'pyproject.toml').exists():
            repo_root = str(_c)
            break
if not repo_root:
    raise RuntimeError('Could not find repo root')

os.chdir(repo_root)
for _path in [repo_root, os.path.join(repo_root, 'src')]:
    if _path not in sys.path:
        sys.path.insert(0, _path)

# Install project + dependencies into the running kernel (instant if already done)
uv_bin = shutil.which('uv') or os.path.expanduser('~/.local/bin/uv')
_r = subprocess.run(
    [uv_bin, 'pip', 'install', '--quiet', '--prerelease', 'allow',
     '--python', sys.executable, '-e', '.'],
    cwd=repo_root, capture_output=True, text=True)
if _r.returncode != 0:
    print(f'WARNING: uv pip install failed:\n{_r.stderr}')

load_dotenv('secrets.env')

assert os.environ.get('ANTHROPIC_API_KEY'), (
    'ANTHROPIC_API_KEY not set. Get a key at https://console.anthropic.com '
    'and set it: export ANTHROPIC_API_KEY=sk-ant-your_key_here'
)
print('API key configured.')

## Ask a Question About Any Dataset

Point the agent at a data directory and ask a natural language question. The agent will explore the files, write code, and return an answer.

We'll use the DABStep financial dataset as an example, but you can point `data_dir` at any folder with CSV/JSON/Parquet files.

In [ ]:
from nat.runtime.loader import load_workflow
from data_explorer_agent.python_executor import _tools as executor_tools
import json, re, glob as glob_module

QA_CONFIG = os.path.join(repo_root, 'generic_qa_agent/config.yml')

def scan_data_dir(data_dir):
    """Scan a data directory and return a summary."""
    data_dir = os.path.abspath(data_dir)
    lines = [f'Data directory: {data_dir}']
    patterns = ['*.csv', '*.json', '*.jsonl', '*.parquet']
    found = []
    for p in patterns:
        found.extend(glob_module.glob(os.path.join(data_dir, '**', p), recursive=True))
    for fp in sorted(found):
        name = os.path.relpath(fp, data_dir)
        size = os.path.getsize(fp) / 1024
        lines.append(f'  - {name} ({size:.1f} KB)')
    return '\n'.join(lines)

def extract_answer(raw):
    if raw is None: return ''
    raw = str(raw).strip()
    for pattern in [r'"answer"\s*:\s*"([^"]*)"', r'"agent_answer"\s*:\s*"([^"]*)"']:
        m = re.search(pattern, raw)
        if m: return m.group(1).strip()
    try:
        parsed = json.loads(raw)
        if isinstance(parsed, dict):
            for k in ('answer', 'agent_answer', 'result'):
                if k in parsed: return str(parsed[k]).strip()
    except: pass
    return raw

async def ask_qa(data_dir, question):
    """Ask a question about data in the given directory."""
    summary = scan_data_dir(data_dir)
    abs_dir = os.path.abspath(data_dir)
    prompt = f'Available data files:\n{summary}\n\nData directory absolute path: {abs_dir}\n\nQUESTION: {question}'
    
    t0 = time.time()
    async with load_workflow(QA_CONFIG) as wf:
        async with wf.run(prompt) as runner:
            raw = await runner.result()
    elapsed = time.time() - t0
    
    try: await executor_tools['reset_environment']()
    except: pass
    
    return {'answer': extract_answer(raw), 'time': round(elapsed, 1)}

print('QA agent ready.')

## Example: Financial Payments Data

Let's ask some questions about the DABStep dataset -- but remember, this agent has **no `helper.py`**, no examples, and no domain knowledge. It figures everything out from scratch.

In [ ]:
data_dir = os.path.join(repo_root, 'data/context')

questions = [
    'Which country has the most transactions?',
    'What is the average transaction amount by card scheme?',
    'How many merchants are there, and what are their names?',
]

for q in questions:
    print(f'Q: {q}')
    r = await ask_qa(data_dir, q)
    print(f'A: {r["answer"]}')
    print(f'   Time: {r["time"]}s')
    print('-' * 60)

## Try Your Own Data

Upload your own CSV, JSON, or Parquet files to a `your_data/` directory in the repo root, then ask questions about them.

In [ ]:
# Point to your own data directory and ask a question:

# r = await ask_qa(os.path.join(repo_root, 'your_data'), 'What are the top 5 items by revenue?')
# print(f'Answer: {r["answer"]}')